# SpatialNet — SpatialNet Exp E3 Dilated Size

E3: the 3x3 target-size sample spaced two cells apart (40 px instead of 24 px).
Same code, data and settings as the main run; the only difference is the extra train flags: `--roi_size_step 2`.

**Attach two inputs** (any dataset names; the notebook finds them):
1. **Code**: all `.py` files of this repo plus `shufflenetv2.pth`.
2. **Data**: the prepared dataset (`EgoPoint_spatialnet`: `train/ val/ test/`, each with `images/` and `labels/`).

Then **Run all**. Set `QUICK_RUN = True` first to check the whole pipeline in a few minutes
(2 epochs on a small subset); set it to `False` for the real run. Hyper-parameters come from `config.py`.

Code runs from `/tmp`, so only results are saved to `/kaggle/working`:

```
e3_dilated_size/
  best_model.pth, best_model_by_metric.pth, last_checkpoint.pth, final_model.pth, model_fp32/fp16/int8.pth
  training/     history.csv, run_info.json, summary.md, figures/
  evaluation/   metrics.json, summary.md, tables/, predictions/, figures/
```

## 1. Settings and inputs

In [ ]:
import os, sys, shutil, subprocess

QUICK_RUN = False    # True: 2 epochs on a small subset to check the pipeline; False: the real training run
EXTRA_FLAGS = ['--roi_size_step', '2']   # the only difference from the main run
RESUME_FROM = ''     # a previous run's last_checkpoint.pth, to continue past Kaggle's session limit


def find_input(*required):
    """First folder under /kaggle/input that contains every required path."""
    for root, dirs, _ in os.walk('/kaggle/input'):
        dirs[:] = [d for d in dirs if d not in ('images', 'labels')]
        if all(os.path.exists(os.path.join(root, r)) for r in required):
            return root
    raise FileNotFoundError(f'No attached input contains {required}')


CODE_SRC = find_input('train.py', 'test_evaluate.py', 'shufflenetv2.pth')
DATA_DIR = find_input('train/labels', 'val/labels', 'test/labels')
OUTPUT_DIR = '/kaggle/working/' + ('quick_run' if QUICK_RUN else 'e3_dilated_size')
CODE_DIR = '/tmp/spatialnet'   # outside /kaggle/working, so code never lands in the outputs

## 2. Copy code and check the dataset

In [ ]:
shutil.rmtree(CODE_DIR, ignore_errors=True)
os.makedirs(CODE_DIR)
copied = [f for f in sorted(os.listdir(CODE_SRC)) if f.endswith(('.py', '.pth'))]
for f in copied:
    shutil.copy(os.path.join(CODE_SRC, f), CODE_DIR)
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)

import reporting as R
from annotation import copy_subset

if QUICK_RUN:
    subset = '/tmp/spatialnet_quick_data'
    shutil.rmtree(subset, ignore_errors=True)
    copy_subset(DATA_DIR, subset, {'train': 64, 'val': 32, 'test': 32})
    DATA_DIR = subset

R.banner('Setup' + ('  (QUICK RUN: 2 epochs on a small subset)' if QUICK_RUN else ''))
R.kv([('Code', f'{CODE_SRC} -> {CODE_DIR}  ({len(copied)} files)'),
      ('Data', DATA_DIR),
      ('Output', OUTPUT_DIR),
      ('Resume from', RESUME_FROM or '-')])
R.section('Dataset (every label file is parsed; a malformed file stops here)')
R.print_dataset_summary(R.dataset_summary(DATA_DIR))

## 3. Train

In [ ]:
cmd = [sys.executable, 'train.py', '--data_dir', DATA_DIR, '--output_dir', OUTPUT_DIR, '--quantize'] + EXTRA_FLAGS
if QUICK_RUN:
    cmd += ['--epochs', '2', '--batch_size', '16']
if RESUME_FROM:
    cmd += ['--resume', RESUME_FROM]
subprocess.run(cmd, check=True, cwd=CODE_DIR)

## 4. Evaluate

In [ ]:
subprocess.run([sys.executable, 'test_evaluate.py',
                '--model_path', os.path.join(OUTPUT_DIR, 'best_model.pth'),
                '--data_dir', DATA_DIR, '--output_dir', os.path.join(OUTPUT_DIR, 'evaluation'),
                '--multihand', '--calibrate_thresholds'], check=True, cwd=CODE_DIR)

## 5. Key figures

All tables are in `evaluation/summary.md` and `training/summary.md`; PDF versions of every figure are next to the PNGs.

In [ ]:
from IPython.display import Image, Markdown, display

for path in ['training/figures/training_curves.png', 'evaluation/figures/confusion_matrix.png',
             'evaluation/figures/iou_success_rate.png', 'evaluation/figures/qualitative_examples.png']:
    full = os.path.join(OUTPUT_DIR, path)
    if os.path.exists(full):
        display(Markdown(f'**{path}**'))
        display(Image(filename=full, width=900))